# GenAI in Software Engineering— The Model Drafts, You Verify

**Module 16 · Lesson 16.4**  
*Netsetos GenAI Engineering Course v2.0*

**In this lesson you will:**
- Generate, Then Verify
- The Bounded Agent Loop
- Context Selection
- AI Code Review
- Test Coverage
- The CI Merge Gate

---

> This notebook is generated from the published lesson HTML. The HTML is the source of truth - do not hand-edit this notebook. Re-run the generator if the lesson updates.


In [ ]:
# Reproducibility - the lesson uses seeded random values throughout

## The one-character bug that reached main

> 💡 **Info**
>
> An AI coding assistant is asked for a token-expiry check and writes a clean, idiomatic function in four seconds. It is almost right: it compares the token’s expiry with a less-than instead of a less-than-or-equal, so a token stays valid for one second past expiry. On a screen full of plausible code, nobody notices; the reviewer skims it, it looks fine, it gets committed. That one-character bug is the whole lesson. An AI’s draft is *fast, fluent, and unverified*, and the engineering job is not to write the code — it is to build the harness that catches the draft’s mistakes before they ship. Over the next seven steps you will build that harness as runnable checks: a test gate that will not accept a draft until it is green, a plan-code-test-fix loop with a cap so a stuck agent cannot loop forever, a context selector that packs the right files into a token budget, a review that ranks findings by severity so one real bug blocks the merge, a coverage check that catches the untested branch where the bug hides, a CI merge gate that treats AI code exactly like human code, and an honest metric that measures durable, accepted code instead of lines emitted. By the end you will not have replaced the engineer — you will have built the discipline that lets one work at the speed of a draft without shipping its bugs.

### 🎯 What you will be able to do after this lesson

- **Apply** a test gate and a branch-coverage check to decide whether an AI-generated function is accepted, and a context selector to pack the right files into a token budget.

- **Analyze** a coding-agent loop for why it needs an iteration cap, and a code-review output for the severity that drives the verdict.

- **Evaluate** an AI change against a CI merge gate, and an AI-assisted team against honest acceptance-and-churn metrics rather than lines of code.

- **Create** a merge gate that treats AI-written code exactly like human code — lint, tests, coverage, review, and human approval, with no auto-merge.

> 📦 **Info**
>
> ✅ Before you startThis is the **second lesson** of Module 16 (Industry Applications), and it carries the same spine as the healthcare opener (Lesson 16.1): the model drafts, and the guardrails make the draft safe to ship. It stands on work you have already done: the agent loop and its stop condition from **Module 11**, the CI pipeline and eval gate from **Lesson 12.6**, the context-window and retrieval ideas from **Module 08**, and the API engineering from **Module 07** (the model call is shown illustratively). Everything runnable here is keyless rule-checking on an illustrative example.

## 60-Second Warm-Up

Flip each card and answer before revealing.

> 🤖 **Analogy**
>
> An AI coding assistant is a **brilliant, fast intern who never says “I am not sure”**. It produces a plausible, confident draft for anything you ask, in seconds — and that is exactly why you never merge its work on sight. You give an intern real tasks and real leverage, but every change they make goes through your tests, your review, and your approval before it ships, because their confidence is not evidence and their fluency is not correctness. The whole discipline of this lesson is the harness you would build around a tireless intern who is right most of the time and wrong without warning: verify before you trust, and make the verification automatic. **Where it breaks down:** an intern learns from your feedback and stops repeating a mistake, while a stateless model will make the same class of error on the next task unless your harness — not the model’s memory — catches it every time, so lean on the guardrails, not on the model improving.

> 📦 **Info**
>
> 🚫 Three misconceptions this lesson kills
> - **“If the generated code looks right, it is right.”** A fluent draft is unverified; it is code only once its tests are green.
> - **“An AI coding agent replaces the engineer.”** It drafts; the engineer architects, reviews, and approves — and a human signs every merge.
> - **“More lines of AI code means more productivity.”** Lines emitted is a vanity metric; acceptance and low churn are the real leverage.

> 💡 **Info**
>
> ⚠️ Anti-patternThe auto-merge bot: an AI opens a pull request, an AI approves it, and it merges to main with no human in the loop and no gate that can say no. It feels like the future and it ships unreviewed bugs at machine speed — the off-by-one from the cold open, multiplied across every PR, landing faster than anyone can read. The whole lesson is the opposite: the AI’s draft is the *start* of the pipeline, and the test gate, the review, the coverage check, the CI gate, and a human approval are what make it safe to merge. Move fast on drafts; never auto-merge from an AI.

---

## 🎯 Concept 1: Generate, Then Verify

### Generate, Then Verify

The first discipline: an AI writes a plausible draft in seconds, but a draft is not code until it passes its tests. Never trust unverified generation. The tests are the contract the draft must clear - and a generated function that has not passed them is a guess, not code. Generation is the cheap part; verification is the part that makes it safe.

Start where every AI-coding failure starts: the fluent draft. Ask a model for a function and it returns something that reads correctly, compiles, and is plausibly right — in seconds. The trap is that *reading correctly is not being correct*. The cold open’s off-by-one compiled fine and passed a skim; what it did not do was pass a test that checked the exactly-at-expiry case. So the rule is simple and non-negotiable: a generated function is a **draft** until its tests are green, and the tests are the contract it must clear. In the worked example, a palindrome function generated from the spec “ignore case and punctuation” is run against five tests; all five pass, so the draft is **accepted** — and when a bug is introduced, a test fails and the draft is **rejected**, its failures sent back to the model. This is the whole spine of the lesson in one step: generation is fast and cheap, verification is what makes it safe, and you never let an unverified draft count as code. The block runs the test gate, keyless.

> ✅ **Analogy**
>
> Generation-then-verification is a **translation you back-translate to check**. A fast translator hands you a fluent paragraph in a language you do not read; it *looks* confident and complete, but you do not publish it on faith — you run it back through an independent check to see if the meaning survived. A test suite is that back-translation for code: it does not care how idiomatic or confident the draft looks, only whether it does what the spec says on every case that matters. A draft that reads beautifully and fails the back-translation is not done, however good it sounds — and a draft nobody back-translated is a guess you have chosen to trust.

An AI generates a function that reads correctly and compiles cleanly. Is it done?

**📝 `01_generate_then_verify.py`** - *runnable*

In [ ]:
# GENERATE, THEN VERIFY: an AI writes a function draft in seconds, but a draft is not done code - it is done when the tests are
# green. Never trust unverified generation. Here the model output is illustrative; the runnable part is the test-suite GATE.
def is_palindrome(s):                                     # pretend this was generated from the spec "ignore case and non-letters"
    clean = [c.lower() for c in s if c.isalpha()]
    return clean == clean[::-1]
TESTS = [                                                  # the spec, encoded as tests: (input, expected)
    ("racecar", True), ("RaceCar", True), ("A man, a plan, a canal: Panama", True),
    ("hello", False), ("", True)]
results = [(inp, is_palindrome(inp), exp) for inp, exp in TESTS]
passed = [r for r in results if r[1] == r[2]]
print("Spec -> generated draft -> run the tests (the gate):")
for inp, got, exp in results:
    ok = got == exp
    print("  [{}] is_palindrome({!r}) = {}  (expected {})".format("PASS" if ok else "FAIL", inp[:22], got, exp))
print()
print("{}/{} tests pass -> {}".format(len(passed), len(TESTS),
      "ACCEPT the generated code" if len(passed) == len(TESTS) else "REJECT - send the failures back to the model"))
print("A generated function that has not passed its tests is a guess, not code. The tests are the contract the draft must clear.")

# Output:
# Spec -> generated draft -> run the tests (the gate):
#   [PASS] is_palindrome('racecar') = True  (expected True)
#   [PASS] is_palindrome('RaceCar') = True  (expected True)
#   [PASS] is_palindrome('A man, a plan, a canal') = True  (expected True)
#   [PASS] is_palindrome('hello') = False  (expected False)
#   [PASS] is_palindrome('') = True  (expected True)
#
# 5/5 tests pass -> ACCEPT the generated code
# A generated function that has not passed its tests is a guess, not code. The tests are the contract the draft must clear.

- A function is generated from a spec (shown illustratively) and then run against 5 tests that encode the spec.
- All 5 pass → **ACCEPT** the draft; introduce a bug and a test fails → **REJECT**, send the failures back.
- The tests are the contract the draft must clear — not how idiomatic or confident it looks.
- A generated function that has not passed its tests is a guess, not code: generation is cheap, verification makes it safe.

#### 💡 What just happened

⚡ What just happened? Generate-then-verify gates a draft behind its tests: all pass -> accept, any fail -> reject and feed the failures back. Tradeoff: none worth keeping - trusting an unverified draft is the whole failure mode. Next: the loop that turns a failing draft into a passing one, with a cap so it cannot run forever.

- A generated draft that only turns accepted when its 5 tests pass
- All pass -> ACCEPT; a bug -> REJECT

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 2: The Bounded Agent Loop

### The Bounded Agent Loop

A coding agent loops plan -> code -> test -> fix, feeding the real test errors back each round, and a HARD iteration cap stops a stuck agent from looping forever. The loop is the pattern behind Claude Code, Aider, and Cursor; the cap is what keeps it from burning your budget on a failure it cannot fix. A bounded loop is the difference between convergence and a runaway.

Rejecting a failing draft (Step 1) is only useful if something fixes it, and that something is the **coding-agent loop**: plan the change, write the code, run the tests, read the failures, fix, and repeat. The loop converges for a real reason — each round feeds the *actual test errors* back to the model, so it is correcting against ground truth, not guessing. This is the pattern behind Claude Code, Aider, and Cursor’s agent. But there is a failure mode the loop must guard against: an agent that *cannot* fix its last failing test will try, fail, try again, and loop forever, burning tokens and wall-clock with nothing to show. So every real agent loop carries a **hard iteration cap**. In the worked example, the failing-test count falls 3, 2, 1, 0 and the loop finishes as DONE at iteration 4; and a stuck agent whose count never reaches zero stops at the cap (5) as CAPPED rather than running forever. The cap is not a performance tuning knob — it is the safety valve that makes the difference between an agent that converges and one that runs away (the same stop-condition discipline you built for agents in Lesson 11.1, here with money on the line). The block runs the bounded loop, keyless.

> 🔥 **Analogy**
>
> The agent loop is a **chef tasting and adjusting a sauce — but not forever**. A good chef tastes, adds salt, tastes again, and each adjustment is guided by the last taste, so the sauce converges on right; that feedback is why the loop works. But if the sauce is broken in a way seasoning cannot fix — curdled, burnt — a chef does not taste-and-adjust into the night; they stop, throw it out, and escalate. The iteration cap is that judgment encoded: keep fixing while each round makes progress, but stop after a bounded number of tries instead of looping on a failure that will not yield. A loop that can never stop is not persistence; it is a runaway with your budget on the stove.

Why does a plan-code-test-fix coding agent need a hard iteration cap?

**📝 `02_agent_loop.py`** - *runnable*

In [ ]:
# THE CODING-AGENT LOOP: plan -> edit -> run tests -> fix -> repeat, with a HARD iteration cap so a stuck agent cannot loop
# forever. The loop is the pattern behind Claude Code, Aider, and Cursor; the cap is what keeps it from burning your budget.
MAX_ITERS = 5
# a deterministic model of the fix loop: tests failing this round, and how many the next fix resolves
failing_by_round = [3, 2, 1, 0]                            # illustrative: the agent fixes some failures each iteration
def agent_loop(failing_by_round, cap):
    for i, failing in enumerate(failing_by_round):
        if i >= cap:
            return {"status": "CAPPED", "iterations": i, "failing": failing}
        print("  iter {}: run tests -> {} failing".format(i + 1, failing))
        if failing == 0:
            return {"status": "DONE", "iterations": i + 1, "failing": 0}
        print("           {} still failing -> feed the errors back, fix, re-run".format(failing))
    return {"status": "CAPPED", "iterations": len(failing_by_round), "failing": failing_by_round[-1]}
print("Coding agent: plan -> code -> test -> fix, capped at {} iterations".format(MAX_ITERS))
r = agent_loop(failing_by_round, MAX_ITERS)
print()
print("result: {} after {} iterations, {} tests failing".format(r["status"], r["iterations"], r["failing"]))
print("The loop converges because every round feeds the real test errors back to the model. The cap is the safety valve:")
print("without it, an agent that cannot fix the last failure re-tries forever - a bounded loop is the difference from a runaway.")

# Output:
# Coding agent: plan -> code -> test -> fix, capped at 5 iterations
#   iter 1: run tests -> 3 failing
#            3 still failing -> feed the errors back, fix, re-run
#   iter 2: run tests -> 2 failing
#            2 still failing -> feed the errors back, fix, re-run
#   iter 3: run tests -> 1 failing
#            1 still failing -> feed the errors back, fix, re-run
#   iter 4: run tests -> 0 failing
#
# result: DONE after 4 iterations, 0 tests failing
# The loop converges because every round feeds the real test errors back to the model. The cap is the safety valve:
# without it, an agent that cannot fix the last failure re-tries forever - a bounded loop is the difference from a runaway.

- The loop runs plan → code → test → fix, feeding the real test errors back each round.
- Failing tests fall 3 → 2 → 1 → 0; at 0 the loop is **DONE** at iteration 4 — it converges because it corrects against ground truth.
- A stuck agent whose count never reaches 0 stops at the cap (5) as **CAPPED**, not looping forever.
- The cap is the safety valve: a bounded loop is the difference between convergence and a runaway.

#### 💡 What just happened

⚡ What just happened? The coding-agent loop converges by feeding real test errors back each round, and a hard iteration cap stops a stuck agent from looping forever (DONE at iter 4 here; CAPPED at 5). Tradeoff: a low cap may stop a loop that needed one more round - tune it, but never remove it. Next: the loop is only as good as the files it can see - context selection.

- The failing-test count driven 3 -> 2 -> 1 -> 0, DONE at iteration 4
- A stuck loop stops at the cap (5) as CAPPED

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 3: Context Selection

### Context Selection

An agent cannot fit the whole repository in its context window, so it must rank files by relevance to the task and pack the most relevant ones into a token budget. Bad context is the top cause of a bad AI edit: an irrelevant file that crowds out the one holding the bug is worse than empty context. Choosing the right files is most of what makes an AI edit land.

The loop from Step 2 can only edit what it can see, and it cannot see everything: a real repository is far larger than any context window. So the hard part of a coding agent is **context selection** — ranking the repo’s files by relevance to the task and packing the most relevant ones that fit a **token budget**. This is retrieval (Module 08) applied to code: Aider builds a repo-map, Cursor retrieves over the codebase, and the quality of that selection largely determines the quality of the edit. The failure mode is subtle: bad context is the top cause of a bad AI edit, and an *irrelevant* file that crowds out the one holding the bug is worse than empty context, because it confidently points the model at the wrong place. In the worked example, for the task “fix the JWT token expiry bug,” five files are scored by relevance; the two auth files (jwt.py at 0.95, middleware at 0.70) are packed within a 6000-token budget, using 4400 of it. The test file clears the relevance floor but does not fit the remaining budget, and a barely-relevant billing file and a weak users model fall below the floor entirely. Catch that nuance — a *relevant* file can still be dropped by *size* — because it is the whole point: context selection is ranking *and* fitting, not ranking alone, and an irrelevant file that crowds out the buggy one is worse than empty context. Choosing the right context under a hard budget is most of what makes an AI code edit land. The block ranks and packs the budget, keyless.

> 🕵️ **Analogy**
>
> Context selection is a **detective pulling the case-relevant files from a vast archive**. A good detective does not haul the entire records room into the interview — they cannot, and it would bury the one folder that matters under a thousand that do not. They rank by relevance to *this* case and carry in the handful that fit on the desk. An agent’s token budget is that desk: finite, and best spent on the files that hold the bug. Dragging in an irrelevant folder is not harmless thoroughness — it crowds out a relevant one and sends the investigation confidently in the wrong direction. The skill is not reading everything; it is choosing the right few.

A coding agent cannot fit the whole repo in context. What is the right move?

**📝 `03_context_selection.py`** - *runnable*

In [ ]:
# CONTEXT SELECTION: an agent cannot read the whole repo - it has a token budget. Rank files by relevance to the task and pack
# the most relevant ones that FIT. Bad context is the top cause of bad AI edits; the ranking is the retrieval step for code.
TASK = "fix the JWT token expiry bug in auth"
BUDGET = 6000; FLOOR = 0.5                                # token budget + a minimum relevance to bother packing a file
FILES = [  # (path, relevance score 0-1, token cost)
    ("auth/jwt.py",        0.95, 1800),
    ("auth/middleware.py", 0.70, 2600),
    ("auth/tests.py",      0.60, 3200),
    ("users/models.py",    0.30, 1500),
    ("billing/stripe.py",  0.05, 2400)]
ranked = sorted(FILES, key=lambda f: f[1], reverse=True)
picked, used, reason = [], 0, {}
for path, rel, cost in ranked:
    if rel < FLOOR:
        reason[path] = "below the {:.1f} relevance floor".format(FLOOR)
    elif used + cost <= BUDGET:
        picked.append(path); used += cost
    else:
        reason[path] = "relevant, but {} tok does not fit the remaining budget".format(cost)
print("Task: {!r}  |  budget: {} tokens, relevance floor {}".format(TASK, BUDGET, FLOOR))
print("Ranked by relevance, packed while each fits the budget:")
for path, rel, cost in ranked:
    mark = "PICK" if path in picked else "skip"
    tail = "" if path in picked else "  <- " + reason[path]
    print("  [{}] {:<20} rel={:.2f} cost={}{}".format(mark, path, rel, cost, tail))
print()
print("selected {} files, {}/{} tokens used.".format(len(picked), used, BUDGET))
print("Note the nuance: auth/tests.py clears the relevance floor (0.60) but does NOT fit the remaining budget after the two")
print("higher-ranked files - a relevant file can still be dropped by size. billing and the users model fall below the floor.")
print("Choosing the right context under a hard budget is most of what makes an AI code edit land.")

# Output:
# Task: 'fix the JWT token expiry bug in auth'  |  budget: 6000 tokens, relevance floor 0.5
# Ranked by relevance, packed while each fits the budget:
#   [PICK] auth/jwt.py          rel=0.95 cost=1800
#   [PICK] auth/middleware.py   rel=0.70 cost=2600
#   [skip] auth/tests.py        rel=0.60 cost=3200  <- relevant, but 3200 tok does not fit the remaining budget
#   [skip] users/models.py      rel=0.30 cost=1500  <- below the 0.5 relevance floor
#   [skip] billing/stripe.py    rel=0.05 cost=2400  <- below the 0.5 relevance floor
#
# selected 2 files, 4400/6000 tokens used.
# Note the nuance: auth/tests.py clears the relevance floor (0.60) but does NOT fit the remaining budget after the two
# higher-ranked files - a relevant file can still be dropped by size. billing and the users model fall below the floor.
# Choosing the right context under a hard budget is most of what makes an AI code edit land.

- For the task “fix the JWT token expiry bug,” five files are scored by relevance to the task.
- Ranked highest-first, files are packed while they clear the 0.5 relevance floor *and* fit the **6000-token budget**.
- auth/jwt.py and middleware are packed (4400/6000); auth/tests.py is relevant (0.60) but does not fit the remaining budget; billing and the users model fall below the floor.
- A relevant file can still be dropped by size, and an irrelevant file that crowds out the buggy one is worse than empty context — selection is ranking *and* fitting.

#### 💡 What just happened

⚡ What just happened? Context selection ranks repo files by relevance and packs the most relevant into a token budget - bad context, not a weak model, is the top cause of a bad AI edit. Tradeoff: a tight budget may drop a file that mattered; too loose and noise crowds out signal. Next: once the agent proposes a change, how a human-plus-AI review decides on it - by severity.

- Five repo files ranked by relevance to the JWT bug
- Packed into a 6000-token budget; jwt.py + middleware picked

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 4: AI Code Review

### AI Code Review

An AI review returns findings ranked by SEVERITY, and severity - not count - drives the verdict. One HIGH correctness or security finding means request-changes, no matter how many style nits are clean. The AI surfaces and ranks the findings; a human reads the HIGH one and decides. AI flags, a person approves.

The agent proposes a change; now something has to judge it, and that is **AI code review** — a pass over the diff that returns findings, each tagged by **severity**. The critical design point is how the verdict is computed: *severity, not count*, drives it. One HIGH finding — a correctness bug, a security hole — forces request-changes, no matter how many low-severity nits are clean, because a single real bug is not outweighed by tidy variable names. In the worked example, a review of the diff returns three findings: a HIGH (the expiry compared with a less-than instead of a less-than-or-equal — the cold open’s bug), a MED (no test at the boundary), and a LOW (an unclear variable name); sorted most-severe-first, the one HIGH yields **request changes**, and only when it is resolved does the verdict flip toward approve. Tools like GitHub Copilot’s PR review, CodeRabbit, and Graphite do exactly this in your pull requests. The rule that keeps it safe is the same one from healthcare: the AI *surfaces and ranks*, and a human *reads the HIGH finding and decides*. AI flags; a person approves. The block runs the severity-ranked review, keyless.

> 📝 **Analogy**
>
> AI review is a **proofreader who marks a manuscript by severity**, not by tally. A good proofreader distinguishes a factual error that makes a sentence false from a missing comma, and a single false statement holds up publication while a dozen comma fixes do not — you do not average them into “mostly fine.” A code review works the same way: one HIGH correctness bug blocks the merge while three style nits never would, because severity is a property of consequence, not of count. And the proofreader does not publish the book — they hand the marked-up pages to an editor who decides. The AI marks and ranks; the human reads the red ink and makes the call.

An AI review returns 1 high, 1 medium, and 3 low findings. What is the verdict?

**📝 `04_code_review.py`** - *runnable*

In [ ]:
# AI CODE REVIEW: a diff comes back as findings ranked by SEVERITY, and the severity decides the verdict. A single HIGH finding
# (a security or correctness bug) means request-changes, no matter how many nits are clean. AI flags; a human still decides.
findings = [  # (file, severity, note) - illustrative output of a review pass over a diff
    ("auth/jwt.py",   "high", "expiry compared with '<' not '<=' - token valid one second past expiry"),
    ("auth/jwt.py",   "med",  "no test covers the exactly-at-expiry boundary"),
    ("auth/util.py",  "low",  "variable name 'x' is unclear")]
order = {"high": 0, "med": 1, "low": 2}
findings.sort(key=lambda f: order[f[1]])
counts = {s: sum(1 for f in findings if f[1] == s) for s in ("high", "med", "low")}
verdict = "REQUEST CHANGES" if counts["high"] else ("COMMENT" if counts["med"] else "APPROVE")
print("AI review of the diff ({} findings), most severe first:".format(len(findings)))
for f, sev, note in findings:
    print("  [{:<4}] {}: {}".format(sev.upper(), f, note))
print()
print("counts: {} high, {} med, {} low  ->  verdict: {}".format(counts["high"], counts["med"], counts["low"], verdict))
print("Severity, not count, drives the verdict: one HIGH correctness bug blocks the merge while three style nits would not.")
print("The AI surfaces and ranks; the human reads the HIGH finding and decides. AI flags, a person approves.")

# Output:
# AI review of the diff (3 findings), most severe first:
#   [HIGH] auth/jwt.py: expiry compared with '<' not '<=' - token valid one second past expiry
#   [MED ] auth/jwt.py: no test covers the exactly-at-expiry boundary
#   [LOW ] auth/util.py: variable name 'x' is unclear
#
# counts: 1 high, 1 med, 1 low  ->  verdict: REQUEST CHANGES
# Severity, not count, drives the verdict: one HIGH correctness bug blocks the merge while three style nits would not.
# The AI surfaces and ranks; the human reads the HIGH finding and decides. AI flags, a person approves.

- The review returns findings tagged by severity, sorted most-severe first: a HIGH, a MED, and a LOW.
- The HIGH is the expiry compared with < instead of <= — the cold open’s off-by-one bug.
- The verdict is **request changes** because there is a HIGH — severity, not count, decides.
- The AI surfaces and ranks; a human reads the HIGH finding and decides. AI flags, a person approves.

#### 💡 What just happened

⚡ What just happened? AI code review ranks findings by severity and lets the severity drive the verdict - one HIGH forces request-changes regardless of the nit count. Tradeoff: an AI reviewer has false positives and misses; it augments the human review, it does not replace it. Next: the HIGH finding pointed at an untested branch - the coverage check that catches it.

- Three findings sorting by severity: 1 high, 1 med, 1 low
- One HIGH -> REQUEST CHANGES

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 5: Test Coverage

### Test Coverage

Generated tests must EXERCISE the code, not just exist. Measure branch coverage and flag the gap, because an untested branch is exactly where the AI's bug hides - and a generated suite that skips the risky branch gives false confidence. “The AI wrote tests” is worthless if the tests avoid the one path that fails.

Step 1 made tests the gate, but that only works if the tests actually *exercise* the code — and generated tests have a specific weakness: they tend to cover the obvious paths and skip the awkward one. So you measure **branch coverage** and flag the gap, because an untested branch is exactly where the AI’s bug hides. In the worked example, a function has five branches (empty input, valid token, expired token, malformed token, and the at-expiry boundary); the generated tests cover four, reaching **80 percent** — and the one uncovered branch is the *at-expiry boundary*, which is precisely the HIGH finding the review flagged in Step 4. That is not a coincidence in the lesson; it is the point: the risky branch and the untested branch are the same branch, and a suite that skips it gives false confidence. “The AI wrote tests” is worthless if the tests avoid the path that fails. Measuring coverage turns “we have tests” into “we have tests, and here is the exact branch nobody is checking” — and the fix is to demand that gap be closed before the change moves on. The block measures branch coverage, keyless.

> 🏢 **Analogy**
>
> Coverage is a **building inspector who checks every room, not just the lobby**. An inspector who signs off after admiring the entrance has told you nothing about the wiring in the back office where the fire will start. Branch coverage is walking into every room: the empty case, the valid case, the malformed case, and — the one that matters — the awkward boundary the builder was tempted to skip. A generated test suite that struts through the lobby and passes is the inspector who never opened the back door; the fault it missed is exactly the one that was hardest to reach. Coverage is the checklist that makes skipping a room visible instead of silent.

The AI-generated tests all pass, but they skip the at-expiry boundary branch. Trust the suite?

**📝 `05_test_coverage.py`** - *runnable*

In [ ]:
# TEST GENERATION + COVERAGE: generated tests must actually EXERCISE the code, not just exist. Measure branch coverage and flag
# the gap - an untested branch is where the AI's bug hides. "The AI wrote tests" is worthless if they skip the risky path.
BRANCHES = ["empty input", "valid token", "expired token", "malformed token", "at-expiry boundary"]
covered_by_tests = {"empty input", "valid token", "expired token", "malformed token"}   # the AI's tests skipped one
covered = [b for b in BRANCHES if b in covered_by_tests]
missing = [b for b in BRANCHES if b not in covered_by_tests]
pct = round(100 * len(covered) / len(BRANCHES))
print("Branch coverage of the generated tests ({} branches):".format(len(BRANCHES)))
for b in BRANCHES:
    print("  [{}] {}".format("x" if b in covered_by_tests else " ", b))
print()
print("coverage: {}/{} branches = {} percent".format(len(covered), len(BRANCHES), pct))
print("uncovered: {}".format(", ".join(missing)))
print("The one skipped branch - the at-expiry boundary - is exactly the HIGH bug the review found. Generated tests that miss")
print("the risky branch give false confidence: measure coverage and demand the gap be closed, do not just count that tests exist.")

# Output:
# Branch coverage of the generated tests (5 branches):
#   [x] empty input
#   [x] valid token
#   [x] expired token
#   [x] malformed token
#   [ ] at-expiry boundary
#
# coverage: 4/5 branches = 80 percent
# uncovered: at-expiry boundary
# The one skipped branch - the at-expiry boundary - is exactly the HIGH bug the review found. Generated tests that miss
# the risky branch give false confidence: measure coverage and demand the gap be closed, do not just count that tests exist.

- The function has 5 branches; the generated tests cover 4 — branch coverage **80 percent**.
- The one uncovered branch is the **at-expiry boundary** — the same branch the review flagged HIGH in Step 4.
- The risky branch and the untested branch are the same branch: a suite that skips it gives false confidence.
- Measure coverage and demand the gap be closed — “the AI wrote tests” is worthless if they avoid the path that fails.

#### 💡 What just happened

⚡ What just happened? Branch coverage catches the untested path where the AI's bug hides - here the at-expiry boundary, the same branch the review flagged HIGH. Tradeoff: high coverage proves the branches are exercised, not that the assertions are meaningful - coverage is necessary, not sufficient. Next: all of it converges at one decision - the CI merge gate.

- Five branches; the generated tests cover 4 = 80 percent
- The at-expiry boundary is uncovered - where the bug hides

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 6: The CI Merge Gate

### The CI Merge Gate

AI-written code goes through the SAME gate as human code - lint, tests, coverage, review with no HIGH open, and a human approval - and it never auto-merges. One unmet check blocks the merge. The gate is where “the AI wrote it” stops mattering and “it passed the checks” starts. Move fast on drafts; never auto-merge from an AI.

Every guardrail so far converges on one decision: does this change merge to main? The **CI merge gate** is where that decision is made, and its defining rule is that AI-written code goes through the *exact same gate* as human code — no special case, no fast lane. The gate checks that lint is clean, all tests pass, branch coverage clears the target, code review has *no HIGH finding open*, and a *human has approved* the merge; one unmet check blocks it. In the worked example, lint and tests pass but three checks are unmet — the uncovered at-expiry branch (Step 5), the open HIGH review finding (Step 4), and the missing human approval — so the decision is **BLOCK**. The single most important line in the gate is the last one: *never auto-merge from an AI*. An AI can open a PR and pass lint and tests, but it does not approve its own work and it does not merge; a human does. This is the same eval-gate discipline from Lesson 12.6, with one addition for AI code — the human approval is not optional. The gate is what lets a team move fast on drafts without shipping an unreviewed bug to main. The block runs the merge gate, keyless.

> 🚪 **Analogy**
>
> The merge gate is the **final airlock before code ships**. An airlock does not care who is trying to pass through — a rookie or the mission commander — it runs the same seal check on everyone, and a single failed seal keeps the door shut. AI-written code is just another crew member at that door: it clears lint, tests, coverage, and review, and a human confirms the pass, or it does not go through. The airlock’s whole value is that it cannot be waved past on reputation or speed; “the AI wrote it and it looked fine” is exactly the kind of trust an airlock is built to refuse. Same door, same checks, for every change.

An AI opened a pull request that passes lint and tests. Should it auto-merge?

**📝 `06_merge_gate.py`** - *runnable*

In [ ]:
# THE CI MERGE GATE: AI-written code goes through the SAME gate as human code - lint, tests, review, and a human approval - and
# NEVER auto-merges. One unmet check blocks the merge. The gate is where "the AI wrote it" stops mattering and "it passed" starts.
GATE = {                                                  # each must hold to merge
    "lint clean":                 True,
    "all tests pass":             True,
    "branch coverage >= target":  False,   # the uncovered at-expiry branch (from the coverage step)
    "code review: no HIGH open":  False,   # the HIGH correctness finding is still open
    "a human approved the merge": False}   # never auto-merge from an AI
unmet = [k for k, ok in GATE.items() if not ok]
print("CI MERGE GATE (same gate for AI and human code):")
for k, ok in GATE.items():
    print("  [{}] {}".format("x" if ok else " ", k))
print()
print("decision: {}".format("MERGE" if not unmet else "BLOCK - {} unmet:".format(len(unmet))))
for u in unmet:
    print("   - " + u)
print("AI code is not special-cased: it clears lint, tests, coverage, and review, and a human approves - or it does not merge.")
print("Never auto-merge from an AI. The gate is what lets you move fast on drafts without shipping an unreviewed bug to main.")

# Output:
# CI MERGE GATE (same gate for AI and human code):
#   [x] lint clean
#   [x] all tests pass
#   [ ] branch coverage >= target
#   [ ] code review: no HIGH open
#   [ ] a human approved the merge
#
# decision: BLOCK - 3 unmet:
#    - branch coverage >= target
#    - code review: no HIGH open
#    - a human approved the merge
# AI code is not special-cased: it clears lint, tests, coverage, and review, and a human approves - or it does not merge.
# Never auto-merge from an AI. The gate is what lets you move fast on drafts without shipping an unreviewed bug to main.

- The merge gate runs the same five checks on AI code as on human code: lint, tests, coverage, review (no HIGH open), human approval.
- Lint and tests pass, but 3 are unmet: the uncovered branch (Step 5), the open HIGH finding (Step 4), and the human approval.
- One unmet check → **BLOCK** — the change does not merge.
- Never auto-merge from an AI: the gate lets you move fast on drafts without shipping an unreviewed bug to main.

#### 💡 What just happened

⚡ What just happened? The CI merge gate runs the same checks on AI code as human code - lint, tests, coverage, review with no HIGH open, and a human approval - and one unmet check (3 here) blocks the merge. Tradeoff: the human-approval step is friction by design; it is the line that keeps an AI from merging its own bug. Next: with all this shipping, how do you honestly measure whether the AI actually helped?

- A five-check gate: lint, tests, coverage, review, human approval
- 3 unmet -> BLOCK; never auto-merge from an AI

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 7: The Honest Metric

### The Honest Metric

“3-5x faster” and lines-of-code are vanity metrics. Measure AI-assisted engineering by ACCEPTANCE (the share of suggested code that survives review) and CHURN (the share of shipped code rewritten or reverted soon after). High acceptance with low churn is real leverage; high churn means you paid the speed back in rework. Lines emitted is calories eaten, not fitness.

The harness is built; the last discipline is measuring whether it actually helped — honestly. The tempting metrics are the worst ones: “3-5x faster” is a feeling, and lines-of-code is a vanity number, because a model can emit ten thousand lines you throw away. The metrics that mean something are **acceptance** (the share of suggested code that survives review and makes it into a PR) and **churn** (the share of shipped code rewritten or reverted soon after). In the worked example, of 1000 suggested lines, 620 are accepted (an acceptance of **62 percent**), and 130 of those are reverted or rewritten within two weeks (a churn of **21 percent** of accepted), leaving 490 durable lines actually kept. High acceptance with low churn is real leverage; high churn means the speed was paid straight back in rework. This matters because the honest evidence is mixed: a widely-discussed 2025 study found experienced developers were, if anything, slightly *slower* with AI on code they knew well, even as they felt faster. Lines emitted is calories eaten; acceptance and churn are fitness. Track the durable outcome, or you optimize a number that does not ship value. The block computes the honest metrics, keyless.

> 🌱 **Analogy**
>
> The honest metric is **measuring a diet by health, not by how much you ate**. “I ate three times as much this month” tells you nothing about whether you are healthier — it might mean the opposite. Lines of AI-generated code is that plate-count: a big, satisfying number that can just as easily be waste as nourishment. Acceptance and churn are the actual health markers — how much of what you produced was good enough to keep, and how much you had to undo. A team generating enormous volumes of code that churns is eating a lot and getting weaker. Measure the outcome that lasts, not the quantity that passed through.

Your AI tool generated three times more lines this quarter. Is the team more productive?

**📝 `07_honest_metrics.py`** - *runnable*

In [ ]:
# THE HONEST METRIC: "3-5x faster" is a vanity claim. Measure AI-assisted engineering by ACCEPTANCE (how much suggested code
# survives review) and CHURN (how much shipped code is rewritten soon after). Lines generated is calories eaten, not fitness.
suggested_lines = 1000
accepted_lines  = 620      # survived review and made it into a PR
reverted_lines  = 130      # shipped, then rewritten or reverted within two weeks (rework)
acceptance = round(100 * accepted_lines / suggested_lines)
churn      = round(100 * reverted_lines / accepted_lines)
net_kept   = accepted_lines - reverted_lines
print("AI-assisted engineering, measured honestly:")
print("  suggested lines: {}".format(suggested_lines))
print("  accepted (survived review): {}  -> acceptance {} percent".format(accepted_lines, acceptance))
print("  reverted/rewritten soon after: {}  -> churn {} percent of accepted".format(reverted_lines, churn))
print("  net durable lines kept: {}".format(net_kept))
print()
print("Lines generated says nothing - a model can emit 10,000 lines of code you throw away. Acceptance and churn measure whether")
print("the assistance actually helped: high acceptance with low churn is real leverage; high churn means you paid the speed back")
print("in rework. Track the durable outcome, not the raw output, or you optimize for a number that does not ship value.")

# Output:
# AI-assisted engineering, measured honestly:
#   suggested lines: 1000
#   accepted (survived review): 620  -> acceptance 62 percent
#   reverted/rewritten soon after: 130  -> churn 21 percent of accepted
#   net durable lines kept: 490
#
# Lines generated says nothing - a model can emit 10,000 lines of code you throw away. Acceptance and churn measure whether
# the assistance actually helped: high acceptance with low churn is real leverage; high churn means you paid the speed back
# in rework. Track the durable outcome, not the raw output, or you optimize for a number that does not ship value.

- Of 1000 suggested lines, 620 are accepted (survive review) — acceptance **62 percent**.
- Of those, 130 are reverted or rewritten soon after — churn **21 percent** of accepted; 490 durable lines are kept.
- Lines emitted says nothing; a model can emit code you throw away. Acceptance and low churn are the real leverage.
- The 2025 evidence is mixed (experienced devs sometimes slower while feeling faster) — track the durable outcome, not raw output.

#### 💡 What just happened

⚡ What just happened? The honest metric measures AI-assisted engineering by acceptance (survives review) and churn (rewritten soon after), not by lines generated - durable, accepted code is the leverage. Tradeoff: acceptance and churn are harder to measure than lines, but a vanity metric that is easy to game is worse than a real one that takes work. That is the lesson: the model drafts, and the harness - tests, loop cap, context, review, coverage, gate, and honest metrics - is what makes the draft ship value.

- 1000 suggested lines -> 620 accepted (62 percent), 130 churned (21 percent)
- Net durable 490; lines is calories, not fitness

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

## Putting It Together

> ✅ **Info**
>
> 🧠 The whole picture — the harness around the model
> AI-assisted engineering is not the model; it is the model plus a harness, and this lesson built that harness as runnable checks. **Generate, then verify** — a draft is code only once its tests are green (Step 1). **Bound the loop** — plan-code-test-fix with a cap so a stuck agent cannot run away (Step 2). **Select the context** — rank files by relevance and pack the budget (Step 3). **Review by severity** — one HIGH finding blocks the merge (Step 4). **Measure coverage** — the untested branch is where the bug hides (Step 5). **Gate the merge** — the same checks as human code, plus a human approval, and never auto-merge (Step 6). And **measure honestly** — acceptance and churn, not lines of code (Step 7). Watch the off-by-one from the cold open thread the whole way through: the review flags it HIGH, the coverage check shows its branch is untested, and the merge gate blocks on it — one bug, caught by six guardrails. Each guardrail stands on earlier work: the agent loop (Module 11), the CI gate (Lesson 12.6), context and retrieval (Module 08). The single sentence to carry forward: an AI does not remove the engineering; it moves the engineer’s job from writing the draft to verifying it.

**📦 **The AI-coding-guardrail picker****

| Guardrail | What it protects against | What it produces | Who owns it |
|---|---|---|---|
| Test gate | Trusting an unverified draft | An accept / reject on the generated code | The engineer |
| Bounded agent loop | A stuck agent looping forever | A converged fix, or a capped stop | The agent framework |
| Context selector | A confident edit in the wrong file | The relevant files within a token budget | The agent / retrieval layer |
| Code review | A real bug hidden among nits | Severity-ranked findings + a verdict | AI flags, a human decides |
| Coverage check | Tests that skip the risky branch | A coverage percent + the uncovered branch | The engineer |
| CI merge gate | An unreviewed AI change reaching main | A merge / block decision + human approval | CI + an accountable human |

> 📦 **Info**
>
> 🔐 One AI-specific risk to name: generated-code securityEverything above treats an AI draft like any other code, but AI-generated code has failure modes of its own that the review (Step 4) and the merge gate (Step 6) must be told to catch. Two are worth naming. **Hallucinated or unpinned dependencies:** a model will confidently `import` a package that does not exist or is not pinned to a version, and attackers register those hallucinated names to get their code installed — a supply-chain attack sometimes called “slopsquatting.” Treat any new or unpinned dependency an AI adds as a HIGH review finding until a human verifies it exists, is the right package, and is pinned. **Hardcoded secrets and injected vulnerabilities:** a model may paste an API key straight into the source or reach for an unsafe pattern (a raw SQL string, a missing auth check); a secret scanner and a security linter belong in the same gate as the tests. The rule is the one from Lesson 15's security work, applied here: an AI writing your code widens the attack surface, so the harness has to include a security lens, not just a correctness one.

> 📦 **Info**
>
> ➡️ Where this goes nextSoftware engineering is the industry the learner lives in, so it is the most concrete place to see the “model plus guardrails” pattern: verify before you trust, bound the loop, select the context, review by severity, cover the risky branch, gate the merge, and measure the durable outcome. It is the same shape as healthcare’s safety system in Lesson 16.1, applied to code. The next lesson carries the pattern to the people who are not developers at all: in Lesson 16.5 you will see how no-code and low-code tools let a PM, analyst, or marketer build with AI, and where these same guardrails still have to live even when nobody is writing the plumbing. Carry one sentence forward: an AI does not remove the engineering; it moves the engineer’s job from writing the draft to verifying it. The reference points here are [SWE-bench](https://www.swebench.com/) and the honest-metrics [productivity evidence](https://metr.org/blog/2025-07-10-early-2025-ai-experienced-os-dev-study/). Build the harness.

*Practice exercises are in the companion practice notebook.*

---

## 🎓 Summary

You've completed the practical part of **GenAI in Software Engineering— The Model Drafts, You Verify**.

### Next steps:
1. Re-run cells with different parameters to build intuition
2. Try the practice exercises (see `practice-lab/practice-lab-16_4.ipynb` if available)
3. Review the full HTML lesson for the complete narrative

*Generated from `lesson-16.4.html` - regenerate this notebook whenever the source HTML is updated.*
